In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss
import albumentations as A
from albumentations.pytorch import ToTensorV2
import rasterio
from rasterio.plot import show
from matplotlib import pyplot as plt
from tqdm import tqdm
from torchmetrics.classification import MulticlassJaccardIndex
import warnings
warnings.filterwarnings('ignore')

Блок 2: Конфигурация и настройки

In [ ]:
class Config:
    # Пути к данным (нужно адаптировать под вашу структуру)
    TRAIN_IMAGE_DIR = 'train/train/image'
    TRAIN_MASK_DIR = 'train/train/mask'
    VAL_IMAGE_DIR = 'train/val/image'
    VAL_MASK_DIR = 'train/val/mask'
    
    # Параметры модели
    ENCODER = 'timm-efficientnet-b5'
    ENCODER_WEIGHTS = 'imagenet'
    CLASSES = 4  # фон + 3 класса
    ACTIVATION = 'softmax2d'
    
    # Параметры обучения
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    BATCH_SIZE = 4  # уменьшить если не хватает памяти
    LR = 1e-4
    EPOCHS = 50
    NUM_WORKERS = 2
    
    # Аугментации
    IMAGE_SIZE = 512
    
config = Config()

Блок 3: Определение датасета с улучшенной аугментацией

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None, preprocessing=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.preprocessing = preprocessing
        self.images = sorted([f for f in os.listdir(images_dir) if f.endswith('.tif')])
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image_name = self.images[idx]
        image_path = os.path.join(self.images_dir, image_name)
        mask_path = os.path.join(self.masks_dir, image_name)
        
        # Чтение изображения
        with rasterio.open(image_path) as img:
            image = img.read().transpose(1, 2, 0).astype('float32')
        
        # Чтение маски
        with rasterio.open(mask_path) as msk:
            mask = msk.read(1).astype('float32')
        
        # Применение аугментаций
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        
        # Нормализация
        if self.preprocessing:
            preprocessed = self.preprocessing(image=image, mask=mask)
            image = preprocessed['image']
            mask = preprocessed['mask']
        
        # Приведение маски к правильному формату
        mask = mask.long()
        
        return image, mask

# Мощные аугментации для тренировочных данных
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.2, scale_limit=0.2, rotate_limit=45, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.OneOf([
        A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MotionBlur(blur_limit=(3, 7), p=1.0),
    ], p=0.5),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Минимальные преобразования для валидационных данных
val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

Блок 4: Определение модели и функций потерь

In [ ]:
# Создание модели
def create_model():
    model = smp.DeepLabV3Plus(
        encoder_name=config.ENCODER,
        encoder_weights=config.ENCODER_WEIGHTS,
        classes=config.CLASSES,
        activation=config.ACTIVATION,
    )
    return model.to(config.DEVICE)

# Комбинированная функция потерь
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.dice_loss = DiceLoss(mode='multiclass', from_logits=False)
        self.focal_loss = FocalLoss(mode='multiclass', gamma=gamma, alpha=None)
        
    def forward(self, inputs, targets):
        dice = self.dice_loss(inputs, targets)
        focal = self.focal_loss(inputs, targets)
        return self.alpha * dice + (1 - self.alpha) * focal

# Функция для вычисления IoU
def calculate_iou(preds, targets):
    iou_metric = MulticlassJaccardIndex(num_classes=config.CLASSES, ignore_index=0)
    return iou_metric(preds.argmax(dim=1), targets)

Блок 5: Обучение и валидация

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, scheduler, loss_fn, epochs):
    best_iou = 0.0
    train_losses = []
    val_losses = []
    val_ious = []
    
    for epoch in range(epochs):
        # Обучение
        model.train()
        epoch_train_loss = 0.0
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for images, masks in progress_bar:
            images = images.to(config.DEVICE)
            masks = masks.to(config.DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            loss.backward()
            optimizer.step()
            
            epoch_train_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())
        
        train_losses.append(epoch_train_loss / len(train_loader))
        
        # Валидация
        model.eval()
        epoch_val_loss = 0.0
        epoch_val_iou = 0.0
        
        with torch.no_grad():
            val_progress = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
            for images, masks in val_progress:
                images = images.to(config.DEVICE)
                masks = masks.to(config.DEVICE)
                
                outputs = model(images)
                loss = loss_fn(outputs, masks)
                iou = calculate_iou(outputs, masks)
                
                epoch_val_loss += loss.item()
                epoch_val_iou += iou.item()
                
                val_progress.set_postfix(loss=loss.item(), iou=iou.item())
        
        avg_val_loss = epoch_val_loss / len(val_loader)
        avg_val_iou = epoch_val_iou / len(val_loader)
        
        val_losses.append(avg_val_loss)
        val_ious.append(avg_val_iou)
        
        # Обновление learning rate
        scheduler.step(avg_val_loss)
        
        # Сохранение лучшей модели
        if avg_val_iou > best_iou:
            best_iou = avg_val_iou
            torch.save(model.state_dict(), 'best_model.pth')
            print(f'Новая лучшая модель сохранена с IoU: {best_iou:.4f}')
        
        print(f'Epoch {epoch+1}/{epochs} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {avg_val_loss:.4f} | Val IoU: {avg_val_iou:.4f}')
    
    return train_losses, val_losses, val_ious

Блок 6: Подготовка данных и обучение

In [ ]:
# Создание датасетов и загрузчиков
train_dataset = SegmentationDataset(
    config.TRAIN_IMAGE_DIR, 
    config.TRAIN_MASK_DIR, 
    transform=train_transform
)

val_dataset = SegmentationDataset(
    config.VAL_IMAGE_DIR, 
    config.VAL_MASK_DIR, 
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

# Инициализация модели, оптимизатора и планировщика
model = create_model()
optimizer = optim.Adam(model.parameters(), lr=config.LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)
loss_fn = CombinedLoss(alpha=0.7)

# Запуск обучения
train_losses, val_losses, val_ious = train_model(
    model, train_loader, val_loader, optimizer, scheduler, loss_fn, config.EPOCHS
)

Блок 7: Визуализация результатов и прогнозирование

In [ ]:
# Функция для визуализации прогнозов
def visualize_predictions(model, dataset, num_samples=5):
    model.eval()
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples*5))
    
    for i, idx in enumerate(indices):
        image, true_mask = dataset[idx]
        
        with torch.no_grad():
            input_tensor = image.unsqueeze(0).to(config.DEVICE)
            prediction = model(input_tensor)
            pred_mask = torch.argmax(prediction.squeeze(), dim=0).cpu().numpy()
        
        # Оригинальное изображение (только первые 3 канала для визуализации)
        orig_image = image[:3].permute(1, 2, 0).numpy()
        orig_image = (orig_image - orig_image.min()) / (orig_image.max() - orig_image.min())
        
        axes[i, 0].imshow(orig_image)
        axes[i, 0].set_title('Изображение')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(true_mask, cmap='jet')
        axes[i, 1].set_title('Истинная маска')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(pred_mask, cmap='jet')
        axes[i, 2].set_title('Предсказанная маска')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Визуализация нескольких примеров
visualize_predictions(model, val_dataset)

# Построение графиков обучения
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Loss during Training')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_ious, label='Val IoU', color='green')
plt.title('IoU during Validation')
plt.xlabel('Epoch')
plt.ylabel('IoU')
plt.legend()

plt.tight_layout()
plt.show()

Блок 8: Постобработка и сохранение результатов

In [ ]:
# Функция для постобработки масок
def postprocess_mask(mask):
    import cv2
    from scipy import ndimage
    
    processed_mask = np.zeros_like(mask)
    
    for class_id in range(1, config.CLASSES):
        class_mask = (mask == class_id).astype(np.uint8)
        
        # Морфологические операции
        kernel = np.ones((3, 3), np.uint8)
        class_mask = cv2.morphologyEx(class_mask, cv2.MORPH_CLOSE, kernel)
        class_mask = cv2.morphologyEx(class_mask, cv2.MORPH_OPEN, kernel)
        
        # Удаление маленьких объектов
        labeled, num_features = ndimage.label(class_mask)
        sizes = ndimage.sum(class_mask, labeled, range(1, num_features + 1))
        
        for j in range(1, num_features + 1):
            if sizes[j - 1] < 100:  # Минимальный размер объекта
                class_mask[labeled == j] = 0
        
        processed_mask[class_mask == 1] = class_id
    
    return processed_mask

# Функция для прогнозирования на тестовых данных
def predict_test_set(model, test_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    test_images = sorted([f for f in os.listdir(test_dir) if f.endswith('.tif')])
    
    model.eval()
    for image_name in tqdm(test_images, desc='Processing test images'):
        image_path = os.path.join(test_dir, image_name)
        
        with rasterio.open(image_path) as img:
            image = img.read().transpose(1, 2, 0).astype('float32')
            meta = img.meta.copy()
        
        # Преобразование и нормализация
        transform = A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
        
        transformed = transform(image=image)
        image_tensor = transformed['image'].unsqueeze(0).to(config.DEVICE)
        
        # Прогнозирование
        with torch.no_grad():
            prediction = model(image_tensor)
            pred_mask = torch.argmax(prediction.squeeze(), dim=0).cpu().numpy()
        
        # Постобработка
        processed_mask = postprocess_mask(pred_mask)
        
        # Сохранение результата
        meta.update({
            'count': 1,
            'dtype': 'uint8',
            'nodata': 0,
            'compress': 'lzw'
        })
        
        output_path = os.path.join(output_dir, image_name)
        with rasterio.open(output_path, 'w', **meta) as dst:
            dst.write(processed_mask.astype('uint8'), 1)
    
    print(f'Результаты сохранены в {output_dir}')

# Запуск прогнозирования на тестовом наборе
# predict_test_set(model, '/content/test/images', '/content/test/predictions')

Этот код представляет собой комплексное решение, которое включает:

Современную архитектуру DeepLabV3+ с эффективным энкодером

Мощные аугментации данных с Albumentations

Комбинированную функцию потерь (Dice + Focal)

Планировщик обучения с уменьшением скорости при плато

Расширенную визуализацию и мониторинг обучения

Постобработку масок для улучшения качества

Полный пайплайн для обработки тестовых данных